# H3 力加減を探す：試し掴みと潰れの見張り

**ねらい**：作品のその 1 は、箱を **弱い力から少しずつ強めて試し掴み** し、持ち上がった最小の力（力加減）を見つける。
そのあいだ、指が箱を挟む力を見張り、**箱の強さ**（段ボール 15 N、発泡スチロール 6 N）を超えたら「潰れた」とする。
ここでは、
- 試す力の **並び**（弱い順／強めから／いきなり最大）
- 潰れの判定に使う力（**2 本の指の小さいほう** か、足したものか）
- 見つけた力加減のまま運ぶか、**余裕** を足すか

を書き換えて、作品の値の理由を確かめる。H1c の道具をそのまま使う。

## 1. 準備

In [ ]:
import os, time
if "google.colab" in str(get_ipython()):
    !apt-get -qq install -y libosmesa6 fonts-noto-cjk > /dev/null
    !pip -q install genesis-world==1.4.3
    if not os.path.exists("SO-ARM100"):
        !git clone -q https://github.com/TheRobotStudio/SO-ARM100.git && cd SO-ARM100 && git checkout -q a758567
    MODEL = "/content/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
else:   # 手元の Docker（答え合わせ用）
    MODEL = "/work/third_party/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
import numpy as np, matplotlib.pyplot as plt, genesis as gs
# グラフの日本語：Colab は matplotlib を先に読み込んでいるので、入れたフォントを直接登録する（名前だけ指定しても見つからない）
from matplotlib import font_manager
for _f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
    if os.path.exists(_f):
        font_manager.fontManager.addfont(_f)
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]

## 2. 道具と場面（H1c と同じ）

In [ ]:
import math
OFF = np.array([-0.0079, -0.000218, -0.0981]); FOLD = np.array([0.0, -1.7, 1.6, 1.2, 0.0, 0.0])
OPEN, CLOSE, PRE, JAW = 1.2, -0.17, 0.5, 0.019    # 指の角度（開く／閉じきる／速く閉じる 1 段目の止め所）、目印 → 箱の中心
def rot(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])
def quat_of(R):
    t = np.trace(R)
    if t > 0:
        S = math.sqrt(t + 1.0) * 2; q = [0.25 * S, (R[2, 1] - R[1, 2]) / S, (R[0, 2] - R[2, 0]) / S, (R[1, 0] - R[0, 1]) / S]
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        S = math.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2; q = [(R[2, 1] - R[1, 2]) / S, 0.25 * S, (R[0, 1] + R[1, 0]) / S, (R[0, 2] + R[2, 0]) / S]
    elif R[1, 1] > R[2, 2]:
        S = math.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2; q = [(R[0, 2] - R[2, 0]) / S, (R[0, 1] + R[1, 0]) / S, 0.25 * S, (R[1, 2] + R[2, 1]) / S]
    else:
        S = math.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2; q = [(R[1, 0] - R[0, 1]) / S, (R[0, 2] + R[2, 0]) / S, (R[1, 2] + R[2, 1]) / S, 0.25 * S]
    q = np.array(q); return q / np.linalg.norm(q)
def Ry(a): return np.array([[math.cos(a), 0, math.sin(a)], [0, 1, 0], [-math.sin(a), 0, math.cos(a)]])

gs.init(backend=gs.cpu, logging_level="warning")
sc = gs.Scene(sim_options=gs.options.SimOptions(dt=0.01, substeps=10),
              rigid_options=gs.options.RigidOptions(box_box_detection=True, friction_cone=gs.friction_cone.elliptic, impratio=10.0),
              show_viewer=False)
sc.add_entity(gs.morphs.Plane())
A = sc.add_entity(gs.morphs.MJCF(file=MODEL, decompose_robot_error_threshold=0.0), material=gs.materials.Rigid(friction=1.0))
RB = 0.22
box = sc.add_entity(gs.morphs.Box(size=(0.03,) * 3, pos=(RB, 0, 0.015)), material=gs.materials.Rigid(friction=1.0, rho=1000.0),
                    surface=gs.surfaces.Default(color=(0.72, 0.55, 0.36)))
cam = sc.add_camera(res=(768, 432), pos=(0.22, -0.30, 0.15), lookat=(0.20, 0.0, 0.03), fov=45, GUI=False)
t0 = time.time(); sc.build(); print(f"build {time.time() - t0:.0f} s")
g, upper = A.get_link("gripper"), A.get_link("upper_arm")
FIXED, MOVING = g.idx, A.get_link("moving_jaw_so101_v1").idx
A.set_dofs_position(np.zeros(6)); Rd = Ry(math.pi / 2) @ rot(g.get_quat().cpu().numpy())   # 真下向き（H1b）
FL0 = A.get_dofs_frictionloss().cpu().numpy()

def ik(p):
    return A.inverse_kinematics(link=g, pos=p - Rd @ OFF, quat=quat_of(Rd), init_qpos=A.get_dofs_position(), max_solver_iters=100).cpu().numpy()
def tip():
    return g.get_pos().cpu().numpy() + rot(g.get_quat().cpu().numpy()) @ OFF
def jaws():
    """(固定側の指が押す力, 動く側の指が押す力) [N]"""
    c = A.get_contacts(with_entity=box)
    la, lb = c["link_a"].cpu().numpy(), c["link_b"].cpu().numpy()
    f = np.linalg.norm(c["force_a"].cpu().numpy(), axis=-1)
    if "valid_mask" in c: f = f * c["valid_mask"].cpu().numpy()      # 並列の環境のときだけ付く（空きの欄の印）
    return float((f * ((la == FIXED) | (lb == FIXED))).sum()), float((f * ((la == MOVING) | (lb == MOVING))).sum())
def squeeze():
    return min(jaws())
def tilt():
    q = box.get_quat().cpu().numpy(); return math.degrees(math.acos(max(-1.0, min(1.0, 1 - 2 * (q[1] ** 2 + q[2] ** 2)))))
def set_grip(tau):
    """指のモーターの力の上限 = 握る力の設定 [N·m]（ほかの関節は 3.35）"""
    lim = np.full(6, 3.35); lim[5] = tau; A.set_dofs_force_range(-lim, lim)

cur = [None]
def go(p, grip, n, every=None):
    """目印を p へ、真下向きのまま n step かけて S 字で動かす（H1b の move と同じ）。指の角度は grip。"""
    start = cur[0]
    for k in range(n):
        s = min(1.0, (k + 1) / (0.7 * n)); s = 3 * s * s - 2 * s ** 3
        if k % 5 == 0 or k == n - 1:
            q = ik(start + (p - start) * s); q[5] = grip; A.control_dofs_position(q)
        sc.step()
        if every: every()
    cur[0] = np.array(p, float)
def start(mass, frictionloss=None):
    """初めの状態に戻し、箱の重さを mass [kg] に。指を開いて、箱の上 8 cm に置く"""
    sc.reset(); box.set_mass(mass); set_grip(3.35)
    fl = FL0.copy()
    if frictionloss is not None: fl[5] = frictionloss
    A.set_dofs_frictionloss(fl)
    cur[0] = np.array([RB - JAW, 0, 0.08]); q = ik(cur[0]); q[5] = OPEN
    A.set_dofs_position(q); A.control_dofs_position(q)
    for _ in range(30): sc.step()
def grasp(tau, mass, frictionloss=None):
    """下ろして、指を 2 段階で閉じる（速く 1 段目まで → 力の上限を tau に絞って閉じきる。作品の close_gently）"""
    start(mass, frictionloss)
    go([RB - JAW, 0, 0.02], OPEN, 100)
    go(cur[0], PRE, 40)
    set_grip(tau); go(cur[0], CLOSE, 200)
def show():
    sc.visualizer.update(force=True); plt.figure(figsize=(8, 4.5)); plt.imshow(cam.render(rgb=True)[0]); plt.axis("off"); plt.show()

grasp(0.6, 0.15); print("0.6 N·m で 150 g を挟んだ：固定側 %.2f N、動く側 %.2f N" % jaws()); show()

## 3. 試し掴み
`search(重さ, 試す力の並び)`：試す力ごとに「下ろす → 2 段階で閉じる → 2 cm 持ち上げる」。持ち上がって傾きが 10° 未満なら、その力が **力加減**。
持ち上がらなければ、その場で指を開いて次の力へ。毎 step、挟む力（0.05 秒の平均）を見張り、箱の強さを超えたら「潰れた」で止める。

In [ ]:
GENTLE = [0.2, 0.25, 0.3, 0.4, 0.5, 0.65, 0.8]     # 壊れやすい箱：弱い順（作品）
QUICK = [0.65, 0.8, 1.2, 1.6]                       # 壊れにくい箱：強めから（作品）
STRONG = [3.35]                                     # いきなり最大（わざと失敗 strong_a）
CARDBOARD, FOAM = 15.0, 6.0                         # 箱の強さ：挟む力 [N] の上限

def search(mass, steps, limit=CARDBOARD, one_side=False):
    """力加減を探す。(見つけた力, 試した回数, 挟む力の最大, 潰れた力) を返す。one_side：2 本の指の力を足して判定（わざと失敗）"""
    start(mass)
    st = {"peak": 0.0, "crushed": None, "recent": [], "tau": None}
    def watch():
        ff, fm = jaws()
        f = ff + fm if one_side else min(ff, fm)
        st["recent"] = (st["recent"] + [f])[-5:]            # 0.05 秒（5 step）の平均
        fm5 = float(np.mean(st["recent"])); st["peak"] = max(st["peak"], fm5)
        if fm5 > limit and st["crushed"] is None:
            st["crushed"] = st["tau"]
    found, tries = None, 0
    for tau in steps:
        st["tau"] = tau; set_grip(3.35)
        go([RB - JAW, 0, 0.08], OPEN, 60, watch); go([RB - JAW, 0, 0.02], OPEN, 80, watch)
        go(cur[0], PRE, 40, watch); set_grip(tau); go(cur[0], CLOSE, 200, watch)
        go([RB - JAW, 0, 0.04], CLOSE, 60, watch); go(cur[0], CLOSE, 30, watch)
        tries += 1
        if st["crushed"] is not None:
            break
        if float(box.get_pos().cpu().numpy()[2]) > 0.025 and tilt() < 10:
            found = tau; break
        set_grip(3.35); go(cur[0], OPEN, 50, watch)       # その場で開いてから、次の力で
    return found, tries, st["peak"], st["crushed"]

def report(label, steps, limit=CARDBOARD, one_side=False, masses=(0.03, 0.15, 0.30)):
    for m in masses:
        f, n, peak, crushed = search(m, steps, limit, one_side)
        res = f"潰れた（{crushed} N·m のとき）" if crushed is not None else (f"力加減 {f} N·m" if f else "持てない")
        print(f"{label:10s} {m * 1000:4.0f} g   {res:22s}   試した回数 {n}   挟む力の最大 {peak:5.1f} N")

report("弱い順", GENTLE)

**期待される出力**（手元の実測）：30 g は 1 回目の 0.2 N·m、150 g と 300 g は 2 回目の 0.25 N·m。挟む力の最大は 3〜5 N で、段ボールの強さ 15 N よりずっと小さい。

## ✏️ 変えてみる 1：いきなり最大の力（わざと失敗 `strong_a`）
**予想**：試さずに、最大の力（3.35 N·m）で握ると？

In [ ]:
report("いきなり最大", STRONG)

**期待される出力**：3 つとも 1 回目で「潰れた」。挟む力の最大は 78〜90 N（段ボールの強さ 15 N の 5 倍以上）。

## ✏️ 変えてみる 2：強めから試す
作品では、Gemini が「壊れにくい」と **自信をもって** 答えた箱（金属）だけ、強めの 0.65 N·m から試す（試す回数が減る）。
**予想**：段ボール（15 N）と、発泡スチロール（6 N）では、結果は同じ？

In [ ]:
report("強めから", QUICK, limit=CARDBOARD, masses=(0.15,))
report("強めから", QUICK, limit=FOAM, masses=(0.15,))

**期待される出力**：段ボールは 1 回目（0.65 N·m）で持てる（挟む力 11 N）。発泡スチロールの強さ（6 N）では「潰れた」。
**作品では**：これが H5 のわざと失敗 `trust_ai`。Gemini が発泡スチロールを「石・壊れにくい」と見誤り、自信が中くらいなのに信じると、強めから試して潰す。

## 4. 潰れの判定に使う力（わざと失敗 `one_side`）
潰れるのは、両側から挟んだとき。片側の指が触れただけでは、箱は逃げる。だから判定には **2 本の小さいほう** を使う。
**予想**：2 本を足した力で判定すると、発泡スチロール（6 N）を弱い順に試したときは？

In [ ]:
report("小さいほう", GENTLE, limit=FOAM, masses=(0.15,))
report("足す", GENTLE, limit=FOAM, one_side=True, masses=(0.15,))

**期待される出力**：小さいほうなら 0.25 N·m で持てる（最大 4 N）。足すと最大 9 N になり、**潰れていないのに「潰れた」と判定** する（判定の誤り）。

## 5. 見つけた力加減のまま運ぶ（わざと失敗 `hold_min`）
力加減は「2 cm を静かに持ち上げられた最小の力」。運ぶときは、手先が動いて揺れる。
**予想**：力加減のまま、左右に 3 cm 揺らしながら 4 秒持つと？ 2 倍の力では？

In [ ]:
def carry(mass, tau, seconds=4.0, record=None):
    grasp(tau, mass); go([RB - JAW, 0, 0.06], CLOSE, 100, record)
    for k_ in range(int(seconds * 100)):
        if k_ % 5 == 0:
            q = ik(np.array([RB - JAW + 0.03 * math.sin(k_ / 100 * 2 * math.pi), 0, 0.06])); q[5] = CLOSE; A.control_dofs_position(q)
        sc.step()
        if record: record()
    return float(box.get_pos().cpu().numpy()[2]), tilt()

for m in [0.15, 0.30]:
    f, *_ = search(m, GENTLE)
    for mult in [1.0, 2.0]:
        z, tl = carry(m, f * mult)
        print(f"{m * 1000:4.0f} g   力加減 {f} × {mult:.0f} ＝ {f * mult:.2f} N·m   4 秒後の箱の高さ {z * 100:4.1f} cm   傾き {tl:4.1f}°   {'運べた' if z > 0.04 else '落とした'}")

**期待される出力**：力加減のまま（× 1）では、150 g も 300 g も落とす。2 倍なら運べる。
**作品では**：運ぶ力 ＝ 力加減 × 2。ただし、潰れの上限の 7 割を超えない（段ボールなら 0.7 × 15 N ÷ 14.5 ＝ 0.72 N·m）。余裕をどこまで削れるかは H4 で。

## 6. 映像で確かめる
300 g を力加減のまま揺らして運ぶ（8 step に 1 コマ）。**変えてみる**：`mult=2.0` にして ▶ し直す。

In [ ]:
from IPython.display import HTML
import base64, imageio
def film(mass=0.30, mult=1.0, name="h3.mp4"):
    f, *_ = search(mass, GENTLE)
    frames, n = [], [0]
    def rec():
        n[0] += 1
        if n[0] % 8 == 0: frames.append(cam.render(rgb=True)[0])
    carry(mass, f * mult, seconds=3.0, record=rec)
    imageio.mimsave(name, frames, fps=12)
    return HTML(f'<video controls width=640 src="data:video/mp4;base64,{base64.b64encode(open(name, "rb").read()).decode()}">')
film()

## 確かめる問い
1. 弱い順に試すと回数は増える。それでも、作品が段ボールを弱い順に試すのはなぜ？
2. 潰れの判定に、2 本の指の「小さいほう」を使うのはなぜ？
3. 力加減に余裕を足すのはなぜ？ 足しすぎると何が起きる？

<details><summary>答え</summary>

1. 見た目では中身（重さ）が分からず、強めから試すと、壊れやすい箱を潰す。回数より、潰さないことを取る
2. 箱が潰れるのは両側から挟まれたとき。片側だけ触れているときは箱が逃げるので、足すと、潰れていないのに潰れたと判定する
3. 力加減は静かに持ち上げられた最小の力で、運ぶ途中の揺れには足りない。足しすぎると箱の強さを超えて潰すので、潰れの上限の 7 割で抑える
</details>